In [1]:
import lightning as L
import lightning.pytorch.callbacks as cb
from lightning.pytorch.loggers import WandbLogger
from torch.utils.data import DataLoader

In [2]:
from oxels.datasets import DGPerspectiveDataset
from oxels.models import UNetModel

In [3]:
import torch
torch.set_float32_matmul_precision("medium")

In [4]:
callbacks = [
    cb.ModelCheckpoint("checkpoints", monitor="validation/loss", save_top_k=3, save_weights_only=True),
]

In [5]:
logger = WandbLogger(
    name="UNet Test",
    save_dir="logs",
    project="oxels",
)

In [6]:
training_dataset = DGPerspectiveDataset("datasets", "ColoredMNIST", 32, 32, split="train", domain_split="id", seed=0)
validation_dataset = DGPerspectiveDataset("datasets", "ColoredMNIST", 32, 32, split="val", domain_split="id", seed=0)
ood_dataset = DGPerspectiveDataset("datasets", "ColoredMNIST", 32, 32, domain_split="ood", seed=0)

Found existing ColoredMNIST dataset in datasets, skipping download.
Found existing ColoredMNIST dataset in datasets, skipping download.
Found existing ColoredMNIST dataset in datasets, skipping download.


In [7]:
training_dataloader = DataLoader(training_dataset, batch_size=32, num_workers=16, shuffle=True)
validation_dataloader = DataLoader(validation_dataset, batch_size=32, num_workers=16, shuffle=False)
ood_dataloader = DataLoader(ood_dataset, batch_size=32, num_workers=16, shuffle=False)

In [8]:
trainer_config = dict(
    max_epochs=10,
    gradient_clip_val=1.0,
    gradient_clip_algorithm="value",
    accumulate_grad_batches=1,
)

In [9]:
model = UNetModel(
    in_features=3,
    out_features=64,
    total_steps=trainer_config["max_epochs"] * len(training_dataloader) // trainer_config["accumulate_grad_batches"],
)

In [10]:
trainer = L.Trainer(
    **trainer_config,
    logger=logger,
    callbacks=callbacks,
    accelerator="cuda",
    devices=1,
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [11]:
trainer.fit(
    model=model,
    train_dataloaders=training_dataloader,
    val_dataloaders=ood_dataloader,
)

wandb: Currently logged in as: larskue (larskue-tu-dortmund-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/home/lars/Documents/code/python/oxels/.venv/lib/python3.13/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /home/lars/Documents/code/python/oxels/notebooks/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name     | Type | Params | Mode 
------------------------------------------
0 | backbone | UNet | 888 K  | train
------------------------------------------
888 K     Trainable params
0         Non-trainable params
888 K     Total params
3.554     Total estimated model params size (MB)
114       Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=10` reached.
